<a href="https://colab.research.google.com/github/mik-nn/CGATS_Generator/blob/main/MosaicScaleElements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.2 MB/s eta 0:00:00


In [11]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import mm
from reportlab.pdfbase import pdfdoc

# ---------------------------------------------------------
# Geometry
# ---------------------------------------------------------

MICRO = 2 * mm
MACRO = 8 * mm

PAGE_W = 40 * mm
PAGE_H = 24 * mm

MICRO_W = 12
MICRO_H = 12

MACRO_W = 2
MACRO_H = 3

# ---------------------------------------------------------
# Load FOGRA55 ICC profile
# ---------------------------------------------------------

def load_icc():
    with open("Ref-ECG-CMYKOGV_FOGRA55_TAC300.icc", "rb") as f:
        return f.read()

ICC_DATA = load_icc()

# ---------------------------------------------------------
# Create ICC profile object
# ---------------------------------------------------------

def make_icc_object(c):
    icc_obj = pdfdoc.PDFStream()
    icc_obj.content = ICC_DATA
    icc_obj.dictionary["N"] = pdfdoc.PDFNumber(7)
    icc_obj.dictionary["Alternate"] = pdfdoc.PDFName("DeviceCMYK")
    return c._doc.Reference(icc_obj)

# ---------------------------------------------------------
# Create DeviceN colorspace object
# ---------------------------------------------------------

def make_devicen_colorspace(c, icc_ref):
    arr = pdfdoc.PDFArray()
    arr.add(pdfdoc.PDFName("DeviceN"))

    names = pdfdoc.PDFArray()
    for n in ["Cyan","Magenta","Yellow","Black","Orange","Green","Violet"]:
        names.add(pdfdoc.PDFName(n))
    arr.add(names)

    arr.add(icc_ref)
    return c._doc.Reference(arr)

# ---------------------------------------------------------
# Set DeviceN color
# ---------------------------------------------------------

def set_devicen_color(c, tints):
    t = " ".join(f"{v:.4f}" for v in tints)
    c._code.append(f"/CS1 cs [{t}] scn")

# ---------------------------------------------------------
# Draw rectangle
# ---------------------------------------------------------

def draw_rect(c, x, y, w, h):
    c._code.append(f"{x:.4f} {y:.4f} {w:.4f} {h:.4f} re f")

# ---------------------------------------------------------
# Color definitions
# ---------------------------------------------------------

LIGHT_CMYK = [
    (30,0,10,0),
    (10,25,0,0),
    (5,10,30,5),
    (0,20,15,0),
    (5,5,5,0),
    (4,3,6,2)
]

DARK_CMYK = [
    (80,20,30,10),
    (60,80,10,20),
    (40,50,80,30),
    (30,70,60,40),
    (60,60,60,60),
    (45,50,55,35)
]

# CMYKOGV with physical spacing only
LIGHT_CMYKOGV = [
    (30,0,10,0,20,0,0),
    (10,25,0,0,0,0,20),
    (5,10,30,5,0,20,0),
    (0,20,15,0,20,0,0),
    (5,5,5,0,0,0,5),     # NeutralGray → slight V
    (4,3,6,2,5,0,0)      # BalanceBeige → slight O
]

DARK_CMYKOGV = [
    (80,20,30,10,30,0,0),
    (60,80,10,20,0,0,30),
    (40,50,80,30,0,30,0),
    (30,70,60,40,20,0,0),
    (60,60,60,60,0,0,0),
    (45,50,55,35,0,0,0)
]

# ---------------------------------------------------------
# Tint helpers
# ---------------------------------------------------------

def cmyk_to_tints(cmyk):
    c,m,y,k = cmyk
    return [c/100, m/100, y/100, k/100, 0,0,0]

def cmykogv_to_tints(cmykogv):
    c,m,y,k,o,g,v = cmykogv
    return [c/100, m/100, y/100, k/100, o/100, g/100, v/100]

# ---------------------------------------------------------
# Draw micro and macro blocks (diagonal pattern)
# ---------------------------------------------------------

def draw_micro(c, colors, mode):
    for j in range(MICRO_H):
        for i in range(MICRO_W):
            idx = (i + j) % 6
            vals = colors[idx]
            t = cmyk_to_tints(vals) if mode=="CMYK" else cmykogv_to_tints(vals)
            set_devicen_color(c, t)
            draw_rect(c, i*MICRO, j*MICRO, MICRO, MICRO)

def draw_macro(c, colors, mode):
    offset = 24 * mm
    for j in range(MACRO_H):
        for i in range(MACRO_W):
            idx = (i + j) % 6
            vals = colors[idx]
            t = cmyk_to_tints(vals) if mode=="CMYK" else cmykogv_to_tints(vals)
            set_devicen_color(c, t)
            draw_rect(c, offset + i*MACRO, j*MACRO, MACRO, MACRO)

# ---------------------------------------------------------
# PDF generator
# ---------------------------------------------------------

def generate_pdf(filename, colors, mode):
    c = canvas.Canvas(filename, pagesize=(PAGE_W, PAGE_H))

    icc_ref = make_icc_object(c)
    devicen_ref = make_devicen_colorspace(c, icc_ref)

    c._code.append(f"/CS1 {devicen_ref.id} 0 R def")

    draw_micro(c, colors, mode)
    draw_macro(c, colors, mode)

    c.showPage()
    c.save()

# ---------------------------------------------------------
# Generate all four
# ---------------------------------------------------------

generate_pdf("LightMosaicCMYK.pdf", LIGHT_CMYK, "CMYK")
generate_pdf("DarkMosaicCMYK.pdf", DARK_CMYK, "CMYK")
generate_pdf("LightMosaicCMYKOGV.pdf", LIGHT_CMYKOGV, "CMYKOGV")
generate_pdf("DarkMosaicCMYKOGV.pdf", DARK_CMYKOGV, "CMYKOGV")

print("Generated all four mosaic PDFs with embedded FOGRA55 ICC.")

Generated: LightMosaicCMYK.pdf, DarkMosaicCMYK.pdf, LightMosaicCMYKOGV.pdf, DarkMosaicCMYKOGV.pdf
